In [27]:
import pandas as pd
import json
import ast
import os
import re

# fact_activity

In [28]:
df_dim_currency = pd.read_csv(
    r"..\data\dim_currency.csv",
    sep=";",
    dtype={
        "CURRENCYID": str,
        "EXCHANGE_RATE_TO_BASE": float,
    },
)

df_fact_activity = pd.read_csv(
    r"..\data\fact_activity.csv",
    sep=";",
    dtype={
        "CASINOID": str,
        "PLAYERID": str,
        "GAMEID": str,
        "CURRENCYID": str,
        "TOTALWAGER": float,
        "TOTALPAYOUT": float,
        "TOTALSPINS": int,
    },
    parse_dates=["DATE", "GOLIVE_DATE"],
)

df_fact_activity = df_fact_activity.merge(
    df_dim_currency[["CURRENCYID", "EXCHANGE_RATE_TO_BASE"]],
    on="CURRENCYID",
    how="left",
    validate="many_to_one",
)

if df_fact_activity["EXCHANGE_RATE_TO_BASE"].isna().any():
    raise ValueError("Missing exchange rate for one or more activity currencies")

df_fact_activity[["TOTALWAGER", "TOTALPAYOUT"]] = (
    df_fact_activity[["TOTALWAGER", "TOTALPAYOUT"]]
    .multiply(df_fact_activity["EXCHANGE_RATE_TO_BASE"], axis=0)
)
df_fact_activity = df_fact_activity.drop(
    columns=["CURRENCYID", "EXCHANGE_RATE_TO_BASE"]
)

df_fact_activity.head()

,DATE,CASINOID,PLAYERID,GAMEID,TOTALWAGER,TOTALPAYOUT,TOTALSPINS,GOLIVE_DATE
0,2026-06-22,CASINO_447,PLAYER_1430360,GAME_6666,51093.173411,38587.822502,97,2026-06-04
1,2026-06-22,CASINO_390,PLAYER_1430364,GAME_5036,2544.492073,404.220376,10,2026-05-30
2,2026-06-23,CASINO_390,PLAYER_1430364,GAME_5992,365.940205,0.000000,4,2026-03-22
3,2026-06-23,CASINO_390,PLAYER_1430364,GAME_4161,18758.309093,7104.161712,39,2025-12-20
4,2026-06-22,CASINO_240,PLAYER_1430402,GAME_6012,2722.677154,0.000000,11,2026-06-30


In [29]:
df_dim_game = pd.read_csv(
    r"..\data\dim_game.csv",
    sep=";",
    dtype={
        "GAMEID": str,
        "GAMETYPE": str,
        "THEMES": str,
        "MECHANICS": str,
    },
)


def parse_and_clean_lists(val):
    # 1. Handle absolute nulls/blanks in the CSV
    if pd.isna(val):
        return None

    try:
        # 2. Safely evaluate the string text into a Python list
        parsed = ast.literal_eval(str(val))

        # 3. Check if the parsed list is empty, e.g., []
        if isinstance(parsed, list) and len(parsed) == 0:
            return None

        return parsed
    except (ValueError, SyntaxError):
        # Fallback if there is a parsing error with invalid string formatting
        return None


# Apply the cleaning function to your columns
df_dim_game["THEMES"] = df_dim_game["THEMES"].apply(parse_and_clean_lists)
df_dim_game["MECHANICS"] = df_dim_game["MECHANICS"].apply(parse_and_clean_lists)

def natural_sort_key(column_name):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", column_name)
    ]


# 1. One-hot encode and naturally sort 'THEMES'
themes_exploded = df_dim_game[["GAMEID", "THEMES"]].explode("THEMES")
themes_encoded = (
    pd.get_dummies(themes_exploded["THEMES"], prefix="", prefix_sep="", dtype=bool)
    .groupby(themes_exploded["GAMEID"])
    .max()
)
sorted_theme_cols = sorted(themes_encoded.columns, key=natural_sort_key)
themes_encoded = themes_encoded.reindex(sorted_theme_cols, axis=1).reset_index()

# 2. One-hot encode and naturally sort 'MECHANICS'
mechanics_exploded = df_dim_game[["GAMEID", "MECHANICS"]].explode("MECHANICS")
mechanics_encoded = (
    pd.get_dummies(mechanics_exploded["MECHANICS"], prefix="", prefix_sep="", dtype=bool)
    .groupby(mechanics_exploded["GAMEID"])
    .max()
)
sorted_mech_cols = sorted(mechanics_encoded.columns, key=natural_sort_key)
mechanics_encoded = mechanics_encoded.reindex(sorted_mech_cols, axis=1).reset_index()

# 3. Merge back into the main DataFrame
df_dim_game = df_dim_game.merge(themes_encoded, on="GAMEID", how="left").merge(
    mechanics_encoded, on="GAMEID", how="left"
)

# 4. Fill missing values with False and convert type to Boolean
flag_cols = [c for c in df_dim_game.columns if c.startswith(("THEME_", "MECH_"))]
df_dim_game[flag_cols] = df_dim_game[flag_cols].fillna(False).astype(bool)

# 5. Drop the original columns
df_dim_game = df_dim_game.drop(columns=["THEMES", "MECHANICS"])

df_dim_game.head()

,GAMEID,GAMETYPE,THEME_1,THEME_2,THEME_3,THEME_4,THEME_5,THEME_6,THEME_7,THEME_8,...,MECH_188,MECH_189,MECH_190,MECH_191,MECH_192,MECH_193,MECH_194,MECH_195,MECH_196,MECH_197
0,GAME_11344,TYPE_1,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,GAME_11373,TYPE_2,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,GAME_11317,TYPE_3,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,GAME_11318,TYPE_3,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,GAME_11364,TYPE_4,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [30]:
df_fact_activity = df_fact_activity.merge(
    df_dim_game,
    on="GAMEID",
    how="left",
    validate="many_to_one",
)

df_fact_activity.head()

,DATE,CASINOID,PLAYERID,GAMEID,TOTALWAGER,TOTALPAYOUT,TOTALSPINS,GOLIVE_DATE,GAMETYPE,THEME_1,...,MECH_188,MECH_189,MECH_190,MECH_191,MECH_192,MECH_193,MECH_194,MECH_195,MECH_196,MECH_197
0,2026-06-22,CASINO_447,PLAYER_1430360,GAME_6666,51093.173411,38587.822502,97,2026-06-04,TYPE_38,False,...,False,False,False,False,False,True,False,False,False,False
1,2026-06-22,CASINO_390,PLAYER_1430364,GAME_5036,2544.492073,404.220376,10,2026-05-30,TYPE_38,False,...,False,False,False,False,False,False,False,False,False,False
2,2026-06-23,CASINO_390,PLAYER_1430364,GAME_5992,365.940205,0.000000,4,2026-03-22,TYPE_25,True,...,False,False,False,False,False,False,False,False,False,False
3,2026-06-23,CASINO_390,PLAYER_1430364,GAME_4161,18758.309093,7104.161712,39,2025-12-20,TYPE_38,False,...,False,False,False,False,False,False,False,False,False,False
4,2026-06-22,CASINO_240,PLAYER_1430402,GAME_6012,2722.677154,0.000000,11,2026-06-30,TYPE_38,False,...,True,False,False,False,False,False,False,False,False,False


In [38]:
import duckdb

con = duckdb.connect(r"target\adp_assessment.duckdb", read_only=True)

df_fact_activity = con.sql(
    "select * from fct_activity"
).df()
con.close()

df_fact_activity

,activity_date,casino_id,player_id,game_id,game_type,total_wager_base,total_payout_base,revenue,total_spins,go_live_date,...,mech_188,mech_189,mech_190,mech_191,mech_192,mech_193,mech_194,mech_195,mech_196,mech_197
0,2026-06-22,CASINO_208,PLAYER_2469448,GAME_5831,TYPE_38,8.480957e+04,1.039262e+05,-1.911665e+04,127,2026-05-15,...,0,0,0,0,0,0,0,0,0,0
1,2026-06-22,CASINO_208,PLAYER_2469448,GAME_4735,TYPE_38,1.367404e+04,6.815279e+03,6.858760e+03,56,2026-05-29,...,0,0,0,0,0,0,0,0,0,0
2,2026-06-22,CASINO_208,PLAYER_2469448,GAME_3451,TYPE_38,4.201768e+05,4.424357e+05,-2.225889e+04,514,2026-06-01,...,1,0,0,0,0,0,0,0,0,0
3,2026-06-22,CASINO_208,PLAYER_2469448,GAME_6624,TYPE_38,1.973821e+05,2.243904e+05,-2.700827e+04,200,2026-05-18,...,1,0,0,0,0,0,0,0,0,0
4,2026-06-22,CASINO_208,PLAYER_2469448,GAME_5369,TYPE_38,5.348043e+04,3.652997e+04,1.695046e+04,186,2025-07-31,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
973826,2026-06-22,CASINO_438,PLAYER_537733,GAME_4664,TYPE_38,1.655845e+04,1.496629e+04,1.592159e+03,95,2026-03-24,...,1,0,0,0,0,0,0,0,0,0
973827,2026-06-23,CASINO_438,PLAYER_537735,GAME_5816,TYPE_38,7.085965e+06,1.302729e+07,-5.941321e+06,714,2025-12-09,...,0,0,0,0,0,0,0,0,0,0
973828,2026-06-22,CASINO_438,PLAYER_537741,GAME_5517,TYPE_38,2.628458e+03,2.940282e+03,-3.118239e+02,23,2025-07-29,...,0,0,0,0,0,0,0,1,0,0
973829,2026-06-23,CASINO_438,PLAYER_537741,GAME_3780,TYPE_38,8.477577e+03,9.562068e+03,-1.084491e+03,35,2026-03-28,...,0,0,0,0,0,1,0,0,0,0
